# B3 training notebook — completed and independently verified

The recovered 300-fit B3 run has passed independent comparison. Read
**PQC_IDS_B3_Verification.md**. The next notebook is
**PQC_IDS_Feature_Consistency_Audit.ipynb**, which fits zero models. This B3
notebook is retained for a deliberate future training run.

**Correction:** my earlier description wrongly listed Protocol as excluded.
The archived B3 code excludes Dst Port and Init Fwd Win Byts; Protocol remains.
The completed run used the correct archived code; no rerun is needed.

# PQC IDS: identifier-free B3 reproduction

The main run completed 420 fits with exact principal F1 sequences. A focused
RF diagnostic subsequently reproduced the four sensitivity exceptions through
order-sensitive probability aggregation at one near-tied test record. Both
original result versions and the strict comparison status are preserved.

This notebook reproduces the **archived B3 identifier-free experiment**:
`Dst Port` and `Init Fwd Win Byts` are excluded from C1, C2 and C3. It compares every
saved raw-result field and the summary configuration against the recovered,
manifest-verified B3 records.

**This is a substantial run: 300 fits over ten splits.** The archived B3
runner has one MI threshold (0.05) and four footprints. Its separate MLP
threshold-sweep extension is outside this notebook. Per split and model:
one C1 fit + one MI fit + eight footprint fits = ten fits; 10 × 3 × 10 = 300.

1. Select **Runtime → Run all**.
2. Upload the updated **PQC_IDS_Reproduction_Working_Copy.zip** supplied with
   this notebook, and connect the Drive account containing your CSV files.
3. Return **PQC_IDS_B3_Reproduction_Results.zip**, including any differences or
   failure logs.

The historical model settings, including RF `n_jobs=-1`, are preserved for
this reproduction. Every numerical difference remains visible under the same
screening rules used for the main run. This notebook does not silently adopt
single-worker prediction or replace locked paper results. It creates a new
output folder and does not resume an old checkpoint.

In [ ]:
prior_b3_summary = globals().get("SUMMARY", {})
if globals().get("_PQC_B3_ACTIVE_RUN") or (isinstance(prior_b3_summary, dict)
        and str(prior_b3_summary.get("scope", "")).startswith("Identifier-free B3")
        and prior_b3_summary.get("run_id")):
    raise RuntimeError(
        "A B3 run is already initialized in this runtime. Re-running this cell would change "
        "the selected output folder. Use the final export cell, or the separate "
        "PQC_IDS_B3_Recover_Results.ipynb for the completed run. For a deliberate new "
        "training run, restart the runtime first."
    )

from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import stat
import subprocess
import sys
import tempfile
import traceback
import uuid
import zipfile
from google.colab import files, drive

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
SESSION = Path(tempfile.mkdtemp(prefix="pqc_b3_", dir="/content"))

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("Upload exactly one updated working-copy ZIP.")
archive_path = SESSION / "working_copy.zip"
archive_path.write_bytes(uploaded[zip_names[0]])
del uploaded
extracted = SESSION / "package"
with zipfile.ZipFile(archive_path) as archive:
    if archive.testzip() is not None:
        raise ValueError("The ZIP failed its integrity check. Download the working copy again.")
    for info in archive.infolist():
        path = PurePosixPath(info.filename)
        if path.is_absolute() or ".." in path.parts or stat.S_ISLNK(info.external_attr >> 16):
            raise ValueError("Unexpected archive path: " + info.filename)
    archive.extractall(extracted)
manifest_paths = list(extracted.glob("*/package_manifest.json"))
if len(manifest_paths) != 1:
    raise ValueError("Expected one working-copy package manifest.")
PACKAGE = manifest_paths[0].parent
manifest = json.loads(manifest_paths[0].read_text())
listed = set()
for item in manifest["files"]:
    path = (PACKAGE / item["path"]).resolve()
    if not path.is_relative_to(PACKAGE.resolve()):
        raise ValueError("Invalid manifest path.")
    if not path.is_file() or path.stat().st_size != item["bytes"] or sha256(path) != item["sha256"]:
        raise ValueError("Package file does not match the manifest: " + item["path"])
    listed.add(item["path"])
actual = {p.relative_to(PACKAGE).as_posix() for p in PACKAGE.rglob("*") if p.is_file()}
if actual != listed | {"package_manifest.json"} or len(listed) != len(manifest["files"]):
    raise ValueError("The archive contains unlisted files or duplicate manifest records.")
if not (PACKAGE / "scripts/compare_b3_reproduction.py").is_file():
    raise ValueError("This is an older working copy. Use the ZIP provided with this notebook.")
print("Working-copy files verified:", len(listed))
print("Run identifier:", RUN_ID)
_PQC_B3_ACTIVE_RUN = RUN_ID


## Connect the verified CSV inputs

The default folder is `/content/drive/MyDrive/cicids2018`. Edit `DATA_DIR`
only if the files moved. Input hashes, row counts, class labels and the full
70-feature prepared matrix must match the successful seed-0 run. The archived
B3 runner then removes the two identifiers from all model conditions,
leaving 68 C1/C2 features and 53 KS/MI candidates.

In [ ]:
drive.mount("/content/drive")
DATA_DIR = Path("/content/drive/MyDrive/cicids2018")
DRIVE_OUTPUT = Path("/content/drive/MyDrive/pqc_ids_reproduction_checks") / RUN_ID
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=False)
RUN_OUTPUT = DRIVE_OUTPUT / "results"
TARGET = json.loads((PACKAGE / "environment/b3_known_versions.json").read_text())
SUMMARY = {
    "run_id": RUN_ID,
    "status": "preparing_environment",
    "colab_kernel_python": platform.python_version(),
    "package_sha256": sha256(archive_path),
    "package_manifest_sha256": sha256(PACKAGE / "package_manifest.json"),
    "data_directory": str(DATA_DIR),
    "result_directory": str(DRIVE_OUTPUT),
    "scope": "Identifier-free B3, ten splits and 300 fits; separate MLP threshold sweep excluded",
    "target": TARGET,
}

def save_summary():
    (DRIVE_OUTPUT / "notebook_status.json").write_text(json.dumps(SUMMARY, indent=2) + "\n")

save_summary()
shutil.copy2(PACKAGE / "package_manifest.json", DRIVE_OUTPUT / "working_copy_manifest.json")
shutil.copy2(PACKAGE / "environment/b3_known_versions.json", DRIVE_OUTPUT / "b3_known_versions.json")
shutil.copy2(PACKAGE / "environment/main_requirements.txt", DRIVE_OUTPUT / "main_requirements.txt")
print("Results will be saved to:", DRIVE_OUTPUT)
SUMMARY["RF_numerical_variability_policy"] = "Keep every strict-comparison exception; retain original n_jobs=-1."
save_summary()

## Create the declared environment

The separate interpreter uses Python 3.12.13 and the same 21 package pins as
the completed main run. These are prospective reproduction settings; they
do not establish a complete historical B3 runtime lock. Installation and
runtime records are included in the output.

In [ ]:
RUN_ENV = os.environ.copy()
for name in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV"):
    RUN_ENV.pop(name, None)
RUN_ENV["PYTHONNOUSERSITE"] = "1"
RUN_ENV["UV_PYTHON_INSTALL_DIR"] = str(SESSION / "python_runtimes")
RUN_ENV["UV_CACHE_DIR"] = str(SESSION / "uv_cache")

def run_logged(command, log_name, env):
    with (DRIVE_OUTPUT / log_name).open("w", encoding="utf-8") as log:
        log.write("Command: " + json.dumps([str(s) for s in command]) + "\n")
        log.flush()
        with subprocess.Popen([str(s) for s in command], stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                              text=True, bufsize=1, env=env) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            return process.wait()

ENVIRONMENT_READY = False
VENV = SESSION / "b3_environment"
FIT_PYTHON = VENV / "bin/python"
try:
    tooling = SESSION / "uv_tooling"
    bootstrap = [sys.executable, "-m", "pip", "install", "--target", tooling,
                 "--no-deps", "uv==0.12.15"]
    if run_logged(bootstrap, "01_uv_install.log", RUN_ENV) != 0:
        raise RuntimeError("Could not install the pinned uv tool.")
    uv_env = RUN_ENV.copy()
    uv_env["PYTHONPATH"] = str(tooling)
    uv = [sys.executable, "-m", "uv"]
    create = uv + ["venv", "--python", TARGET["python"], "--managed-python", VENV]
    if run_logged(create, "02_python_setup.log", uv_env) != 0:
        raise RuntimeError("Could not create the requested Python runtime.")
    install = uv + ["pip", "install", "--python", FIT_PYTHON, "--only-binary", ":all:",
                    "-r", PACKAGE / "environment/main_requirements.txt"]
    if run_logged(install, "03_packages_install.log", uv_env) != 0:
        raise RuntimeError("Could not install the declared package versions.")
    ENVIRONMENT_READY = True
    SUMMARY["status"] = "environment_installed"
except Exception as error:
    SUMMARY["status"] = "environment_setup_failed"
    SUMMARY["error"] = str(error)
    (DRIVE_OUTPUT / "setup_traceback.txt").write_text(traceback.format_exc())
    print("Setup failed:", error)
finally:
    save_summary()
print("Environment ready:", ENVIRONMENT_READY)

## Reproduce B3 and compare its records

Progress is printed for every fit. The runner saves raw results after each
completed split and keeps warnings visible. The comparison checks missing
splits, changed feature lists, F1, ROC-AUC, KS/MI outputs, selected counts and
the complete saved configuration.

The recovered original B3 means are reference values, not success targets to
tune against: RF 0.844033 / 0.595444 / 0.843930; XGBoost 0.850695 / 0.751942 /
0.849034; MLP 0.838124 / 0.596486 / 0.806602 (C1 / C2 / C3). A difference
produces `needs_review` and is retained for inspection.

The known RF mechanism can explain some small differences, but no exception
is automatically attributed to it. New differences must be checked.

In [ ]:
if ENVIRONMENT_READY:
    try:
        SUMMARY["status"] = "running_b3"
        save_summary()
        seed0_reference = PACKAGE / "reference/seed0_reproduction_20260920/results"
        command = [FIT_PYTHON, "-u", PACKAGE / "scripts/run_experiment.py",
                   "--stage", "train", "--variant", "b3",
                   "--data-dir", DATA_DIR, "--output", RUN_OUTPUT,
                   "--cache-dir", SESSION / "input_cache",
                   "--expected-environment", PACKAGE / "environment/b3_known_versions.json",
                   "--expected-inputs", seed0_reference / "input_files.json",
                   "--expected-prepared", seed0_reference / "seed0_status.json"]
        returncode = run_logged(command, "04_b3_run.log", RUN_ENV)
        SUMMARY["b3_training_returncode"] = returncode
        status_path = RUN_OUTPUT / "run_record.json"
        run_record = json.loads(status_path.read_text()) if status_path.is_file() else {}
        SUMMARY["training_status"] = run_record.get("status", "no_run_record")
        SUMMARY["fits_completed"] = run_record.get("fits_completed", 0)
        SUMMARY["expected_fits"] = 300
        SUMMARY["status"] = "b3_training_completed" if returncode == 0 else "b3_training_failed"
        raw_path = RUN_OUTPUT / "revised_pipeline_b3_raw.json"
        if raw_path.is_file():
            comparison_output = DRIVE_OUTPUT / "comparison"
            compare_command = [FIT_PYTHON, "-u", PACKAGE / "scripts/compare_b3_reproduction.py",
                               "--actual", raw_path,
                               "--summary", RUN_OUTPUT / "revised_pipeline_b3_summary.json",
                               "--output", comparison_output]
            comparison_returncode = run_logged(compare_command, "05_b3_comparison.log", RUN_ENV)
            SUMMARY["comparison_returncode"] = comparison_returncode
            successful_run = (returncode == 0 and run_record.get("status") == "completed"
                              and run_record.get("variant") == "b3"
                              and run_record.get("expected_fits") == 300
                              and run_record.get("fits_completed") == 300
                              and run_record.get("split_computations_completed") == list(range(10)))
            SUMMARY["status"] = "passed_b3_reproduction" if successful_run and comparison_returncode == 0 else "needs_review"
            means = comparison_output / "b3_headline_means.csv"
            if means.is_file():
                print(means.read_text())
        print("\nB3 run status:", SUMMARY["status"])
    except Exception as error:
        SUMMARY["status"] = "runner_exception"
        SUMMARY["error"] = str(error)
        (DRIVE_OUTPUT / "runner_traceback.txt").write_text(traceback.format_exc())
        print("Runner failed:", error)
    finally:
        save_summary()
else:
    print("Training skipped because the declared environment was not installed.")

## Export the complete handoff

Run this final cell even if training or comparison fails. The result ZIP
contains run status, logs, environment details, raw and summary results,
comparison tables, and file hashes. It excludes raw CSV inputs and install
caches. A copy is also saved to the new Drive output folder.

Return **PQC_IDS_B3_Reproduction_Results.zip**. A completed training run and
a passed numerical comparison are separate findings; neither establishes
physical validity of the approximation or resolves dependence between splits.

The export reads the recorded status from Drive. It refuses a newly initialized
folder with no training record, so re-running the first cells cannot silently
produce a preparation-only results ZIP.

In [ ]:
from datetime import datetime, timezone

status_file = DRIVE_OUTPUT / "notebook_status.json"
if not status_file.is_file():
    raise RuntimeError("The selected folder has no run status. Use PQC_IDS_B3_Recover_Results.ipynb.")
recorded_summary = json.loads(status_file.read_text())
if recorded_summary.get("run_id") != DRIVE_OUTPUT.name or recorded_summary.get("run_id") != RUN_ID:
    raise RuntimeError("Run identity differs from the selected folder. Use the recovery notebook.")
training_record_file = DRIVE_OUTPUT / "results/run_record.json"
training_record = json.loads(training_record_file.read_text()) if training_record_file.is_file() else {}
if not training_record and recorded_summary.get("status") in ("preparing_environment", "environment_installed", "running_b3"):
    raise RuntimeError(
        "This folder contains setup records but no B3 training record. It is not the "
        "completed run. Use PQC_IDS_B3_Recover_Results.ipynb; do not repeat training "
        "just to recover the earlier files."
    )
export_receipt = {
    "exported_utc": datetime.now(timezone.utc).isoformat(),
    "source_run_id": recorded_summary["run_id"],
    "source_directory": str(DRIVE_OUTPUT),
    "recorded_notebook_status": recorded_summary.get("status"),
    "recorded_training_status": training_record.get("status"),
    "recorded_fits_completed": training_record.get("fits_completed", 0),
    "status_source": "notebook_status.json read from Drive; in-memory SUMMARY is not used",
}
(DRIVE_OUTPUT / "export_receipt.json").write_text(json.dumps(export_receipt, indent=2) + "\n")
result_files = sorted(p for p in DRIVE_OUTPUT.rglob("*")
                      if p.is_file() and p.name not in ("PQC_IDS_B3_Reproduction_Results.zip", "result_manifest.json"))
manifest = [{"path": p.relative_to(DRIVE_OUTPUT).as_posix(), "bytes": p.stat().st_size,
             "sha256": sha256(p)} for p in result_files]
manifest_file = DRIVE_OUTPUT / "result_manifest.json"
manifest_file.write_text(json.dumps({"run_id": recorded_summary["run_id"], "files": manifest}, indent=2) + "\n")
result_files.append(manifest_file)
result_zip = SESSION / "PQC_IDS_B3_Reproduction_Results.zip"
with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in result_files:
        archive.write(path, path.relative_to(DRIVE_OUTPUT).as_posix())
with zipfile.ZipFile(result_zip) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("The result archive failed its integrity check.")
saved_zip = DRIVE_OUTPUT / result_zip.name
shutil.copy2(result_zip, saved_zip)
print("Recorded run status:", recorded_summary["status"])
print("Recorded completed fits:", training_record.get("fits_completed", 0))
print("Results saved:", saved_zip)
print("ZIP SHA-256:", sha256(result_zip))
files.download(str(result_zip))